# 02 CoM

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_003_SVC_CoM_Diagnostic.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_CoM_Diagnostic.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)
- [`EXP_003_SVC_Normal_CoM.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)
- [`EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_CoM_Diagnostic.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'aec1ef61996a3c41ff31e06078610277f3f9e90b'}, '05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd810907b08674d11b2e9aa85b504b57e7985393f'}, '05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '67bece0418ff59b8601d67d5fadb39dcc58a359b'}}

## EXP_003_SVC_CoM_Diagnostic

원본: [05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_CoM_Diagnostic.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_CoM_Diagnostic.py)  
Git blob: `aec1ef61996a3c41ff31e06078610277f3f9e90b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC CoM diagnostics; pilot only, never fits full 76,800 rows.

CLI 인자 (정적 추출): `--train`, `--val`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_CoM_Diagnostic.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC CoM diagnostics; pilot only, never fits full 76,800 rows."""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
SVC_SCRIPTS = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'
KNN_SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path[:0] = [str(SVC_SCRIPTS), str(KNN_SCRIPTS), str(ROOT / '03_experiments' / 'model_optimization')]

from EXP_001_SVC_Normal_Baseline import baseline, report
from EXP004_KNN_Normal_CoM import BatchedCoM
from common import fixed_split, pipeline


def log(*args):
    print(*args, flush=True)


def run(train_n, val_n):
    model, scaled, source = baseline()
    if not scaled:
        raise RuntimeError('SVC baseline scaling must be True')
    log('SVC baseline source:', source)
    log('SVC parameters:', model.get_params())
    log('Pilot training:', train_n, 'validation:', val_n)
    log('Official Test is NOT loaded')
    t = time.perf_counter()
    xtr, ytr, xva, yva, sha = fixed_split()
    log('Data loaded in seconds:', round(time.perf_counter() - t, 2))
    trids, _ = train_test_split(np.arange(len(ytr)), train_size=train_n, stratify=ytr, random_state=42)
    vaids, _ = train_test_split(np.arange(len(yva)), train_size=val_n, stratify=yva, random_state=42)
    xt, yt, xv, yv = xtr[trids], ytr[trids], xva[vaids], yva[vaids]
    del xtr, ytr, xva, yva
    log('Pilot shapes:', xt.shape, xv.shape, 'dtype:', xt.dtype)
    t = time.perf_counter()
    com = BatchedCoM()
    centered_train = com.transform(xt)
    log('CoM train seconds:', round(time.perf_counter() - t, 2))
    t = time.perf_counter()
    centered_val = com.transform(xv)
    log('CoM validation seconds:', round(time.perf_counter() - t, 2))
    log('CoM train range:', float(centered_train.min()), float(centered_train.max()))
    log('CoM train finite:', bool(np.isfinite(centered_train).all()))
    log('CoM train zero rows:', int(np.count_nonzero(np.max(np.abs(centered_train), axis=1) == 0)))
    log('CoM train dtype:', centered_train.dtype)
    if not np.isfinite(centered_train).all():
        raise RuntimeError('Non-finite transformed values')
    clf = pipeline(model, scaled)
    log('SVC pilot fit start...')
    t = time.perf_counter()
    clf.fit(centered_train, yt)
    log('SVC fit seconds:', round(time.perf_counter() - t, 2))
    log('Support vectors:', len(clf.named_steps['svc'].support_) if 'svc' in clf.named_steps else 'see pipeline steps below')
    log('Pipeline steps:', list(clf.named_steps))
    t = time.perf_counter()
    pred = clf.predict(centered_val)
    log('SVC predict seconds:', round(time.perf_counter() - t, 2))
    log('Pilot official Validation:', report(yv, pred))
    log('DIAGNOSTIC COMPLETE; no full training and no experiment results overwritten')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train', type=int, default=2000)
    parser.add_argument('--val', type=int, default=800)
    args = parser.parse_args()
    if not 100 <= args.train <= 8000 or not 100 <= args.val <= 3200:
        parser.error('diagnostics restricted to 100..8000 train and 100..3200 validation')
    run(args.train, args.val)


## EXP_003_SVC_Normal_CoM

원본: [05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py)  
Git blob: `d810907b08674d11b2e9aa85b504b57e7985393f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-003: unchanged baseline versus CoM-centered SVC.

Run from the repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py

Uses official Training/Validation and Hand-made development data; never loads Test.
The original SVC and pixel scaler come from the team's baseline code.
The CoM transformer is reused from KNN Normal EXP004 (same MLP implementation).



In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-003: unchanged baseline versus CoM-centered SVC.

Run from the repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM.py

Uses official Training/Validation and Hand-made development data; never loads Test.
The original SVC and pixel scaler come from the team's baseline code.
The CoM transformer is reused from KNN Normal EXP004 (same MLP implementation).
"""
from __future__ import annotations

import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
SVC_SCRIPTS = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'
KNN_SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(SVC_SCRIPTS))
sys.path.insert(0, str(KNN_SCRIPTS))
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))

from EXP_001_SVC_Normal_Baseline import WRITERS, OUT_BASE, baseline, load_writer, report
from EXP004_KNN_Normal_CoM import BatchedCoM
from common import LABELS, fixed_split, pipeline

BASELINE_DIR = OUT_BASE / 'EXP_001_SVC_Normal_Baseline'
OUT = OUT_BASE / 'EXP_003_SVC_Normal_CoM'


def save_csv(filename, rows, fieldnames):
    with (OUT / filename).open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def main():
    baseline_metrics_path = BASELINE_DIR / 'metrics.json'
    baseline_predictions_path = BASELINE_DIR / 'handmade_predictions.csv'
    if not baseline_metrics_path.exists() or not baseline_predictions_path.exists():
        raise FileNotFoundError('SVC EXP-001 result missing. Run EXP_001_SVC_Normal_Baseline.py first.')
    prior = json.loads(baseline_metrics_path.read_text(encoding='utf-8'))
    with baseline_predictions_path.open('r', encoding='utf-8-sig', newline='') as f:
        prior_rows = list(csv.DictReader(f))
    if len(prior_rows) != 2560 or prior['handmade_development']['correct'] != 1576:
        raise RuntimeError('Unexpected EXP-001 predictions or baseline performance')
    if not np.isclose(prior['official_validation']['accuracy'], 0.9766666666666667, atol=1e-10):
        raise RuntimeError('EXP-001 official validation baseline did not match reported results')

    raw_svc, scaled, source = baseline()
    if not scaled:
        raise RuntimeError('The recorded EXP-001 baseline expects scaling=True')
    if prior.get('model_source') != source:
        raise RuntimeError('SVC baseline source differs from EXP-001')
    if prior.get('hyperparameters') != raw_svc.get_params():
        raise RuntimeError('SVC hyperparameters differ from EXP-001')
    x_train, y_train, x_val, y_val, official_sha = fixed_split()
    if len(y_train) != 76800 or len(y_val) != 19200:
        raise RuntimeError('Official split mismatch')
    if prior.get('dataset_sha256') != official_sha:
        raise RuntimeError('Official dataset digest differs from EXP-001')
    handmade = {writer: load_writer(writer) for writer in WRITERS}
    for writer in WRITERS:
        if prior['handmade_sha256'][str(writer)] != handmade[writer][2]:
            raise RuntimeError(f'Writer {writer} digest differs from EXP-001')

    base_pipe = pipeline(raw_svc, True)
    candidate = Pipeline([('center_of_mass', BatchedCoM()), *base_pipe.steps])
    print('EXP-003 SVC Normal: CoM before existing PixelScaler255 and SVC', flush=True)
    print('Official Training=76800, Validation=19200; Test NOT loaded', flush=True)
    print('Using saved EXP-001 predictions as unchanged baseline (no baseline refit)', flush=True)
    t0 = time.perf_counter()
    candidate.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - t0
    print(f'CoM SVC fit_seconds={fit_seconds:.2f}', flush=True)
    prediction_val = candidate.predict(x_val).astype(np.int64)
    val_scores = report(y_val, prediction_val)
    print(f"Official Validation: EXP-001={prior['official_validation']['accuracy']:.6f} CoM={val_scores['accuracy']:.6f}", flush=True)

    writer_results, pred_rows, true_all, before_all, after_all = [], [], [], [], []
    for writer in WRITERS:
        x, y, _ = handmade[writer]
        prediction = candidate.predict(x).astype(np.int64)
        old_rows = [r for r in prior_rows if int(r['writer']) == writer]
        old_rows = sorted(old_rows, key=lambda r: int(r['index']))
        if len(old_rows) != 640 or [int(r['index']) for r in old_rows] != list(range(640)):
            raise RuntimeError(f'Invalid EXP-001 saved predictions for writer {writer}')
        old_true = np.array([LABELS.index(r['true']) for r in old_rows], dtype=np.int64)
        old_pred = np.array([LABELS.index(r['prediction']) for r in old_rows], dtype=np.int64)
        if not np.array_equal(y, old_true):
            raise RuntimeError(f'Original labels have changed for writer {writer}')
        m = report(y, prediction)
        old_m = report(y, old_pred)
        writer_results.append({'writer': writer, 'n': len(y), 'baseline_accuracy': old_m['accuracy'],
                               'com_accuracy': m['accuracy'], 'baseline_macro_f1': old_m['macro_f1'],
                               'com_macro_f1': m['macro_f1'],
                               'delta_accuracy_pp': 100 * (m['accuracy'] - old_m['accuracy'])})
        print(f"Writer {writer}: Baseline={old_m['accuracy']:.6f}, CoM={m['accuracy']:.6f}", flush=True)
        true_all.append(y)
        before_all.append(old_pred)
        after_all.append(prediction)
        pred_rows.extend({'writer': writer, 'index': i, 'true': LABELS[int(yt)],
                          'baseline_prediction': LABELS[int(pb)], 'com_prediction': LABELS[int(pc)],
                          'baseline_correct': int(yt == pb), 'com_correct': int(yt == pc)}
                         for i, (yt, pb, pc) in enumerate(zip(y, old_pred, prediction)))

    yt = np.concatenate(true_all)
    pb = np.concatenate(before_all)
    pc = np.concatenate(after_all)
    baseline_hm = report(yt, pb)
    com_hm = report(yt, pc)
    if baseline_hm['correct'] != 1576 or not np.isclose(baseline_hm['macro_f1'], 0.604038, atol=0.000001):
        raise RuntimeError('EXP-001 saved Hand-made predictions are inconsistent')
    paired = {'corrected': int(np.sum((pb != yt) & (pc == yt))),
              'new_errors': int(np.sum((pb == yt) & (pc != yt))),
              'unchanged_correct': int(np.sum((pb == yt) & (pc == yt))),
              'unchanged_wrong': int(np.sum((pb != yt) & (pc != yt)))}
    OUT.mkdir(parents=True, exist_ok=True)
    save_csv('writer_results.csv', writer_results, list(writer_results[0]))
    save_csv('handmade_predictions.csv', pred_rows, list(pred_rows[0]))
    cm = confusion_matrix(yt, pc, labels=np.arange(16))
    save_csv('handmade_com_confusion_matrix.csv',
             [{'true/pred': LABELS[i], **{LABELS[j]: int(cm[i, j]) for j in range(16)}} for i in range(16)],
             ['true/pred', *LABELS])
    p, r, f, n = precision_recall_fscore_support(yt, pc, labels=np.arange(16), zero_division=0)
    save_csv('handmade_com_per_class.csv',
             [{'class': LABELS[i], 'precision': float(p[i]), 'recall': float(r[i]),
               'f1': float(f[i]), 'support': int(n[i])} for i in range(16)],
             ['class', 'precision', 'recall', 'f1', 'support'])
    summary = {'experiment': 'EXP-003 SVC Normal CoM', 'model_source': source,
               'hyperparameters': raw_svc.get_params(), 'scaling': scaled,
               'com': 'Same CoM implementation as KNN Normal EXP004 / MLP Automatic_Centering_Transformer',
               'official_validation': {'baseline': prior['official_validation'], 'com': val_scores,
                                       'delta_accuracy_pp': 100 * (val_scores['accuracy']-prior['official_validation']['accuracy'])},
               'handmade_development': {'baseline': baseline_hm, 'com': com_hm,
                                        'delta_accuracy_pp': 100 * (com_hm['accuracy']-baseline_hm['accuracy'])},
               'writers': writer_results, 'paired': paired, 'fit_seconds': fit_seconds,
               'official_train_n': len(y_train), 'official_val_n': len(y_val),
               'official_test_used': False,
               'evaluation_role': 'repeated-use development Hand-made; not independent test',
               'dataset_sha256': official_sha}
    (OUT / 'metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    print(f"Hand-made: Baseline={baseline_hm['accuracy']:.6f}, CoM={com_hm['accuracy']:.6f} F1={com_hm['macro_f1']:.6f}", flush=True)
    print(f"Difference: Official={summary['official_validation']['delta_accuracy_pp']:+.4f}pp Hand-made={summary['handmade_development']['delta_accuracy_pp']:+.4f}pp", flush=True)
    print('Paired:', paired, flush=True)
    print('Saved:', OUT, flush=True)


if __name__ == '__main__':
    main()


## EXP_003_SVC_Normal_CoM_SafePilot_FIXED

원본: [05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py)  
Git blob: `67bece0418ff59b8601d67d5fadb39dcc58a359b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-003 safety pilot: fair baseline-versus-CoM comparison.

Run from repo root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot.py --train-size 16000

This is a reduced-data development pilot, NOT a full 76,800-row experiment.
Never reads official Test. Does not overwrite EXP-001/002/003 result folders.

CLI 인자 (정적 추출): `--train-size`, `--val-size`, `--batch-size`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot_FIXED.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-003 safety pilot: fair baseline-versus-CoM comparison.

Run from repo root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_003_SVC_Normal_CoM_SafePilot.py --train-size 16000

This is a reduced-data development pilot, NOT a full 76,800-row experiment.
Never reads official Test. Does not overwrite EXP-001/002/003 result folders.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP004_KNN_Normal_CoM import BatchedCoM
from common import fixed_split, pipeline

OUT_BASE = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'


def say(*items):
    print(*items, flush=True)


def select_stratified(x, y, n, seed=42):
    if n >= len(y):
        return x, y
    ix, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=seed)
    return x[ix], y[ix]


def batched_predict(model, x, name, batch=256):
    output = []
    start = time.perf_counter()
    for i in range(0, len(x), batch):
        p = model.predict(x[i:i + batch]).astype(np.int64)
        output.append(p)
        if (i // batch) % 8 == 0:
            say(f'  {name}: {min(i + batch, len(x))}/{len(x)} predicted')
    return np.concatenate(output), round(time.perf_counter() - start, 2)


def run(args):
    svc, scaling, source = baseline()
    if not scaling:
        raise RuntimeError('Original project SVC baseline must use scaling')
    say('SVC baseline:', source)
    say('SVC params:', svc.get_params())
    say('SAFE PILOT ONLY. Official Test not loaded.')
    t = time.perf_counter()
    xtr, ytr, xval, yval, sha = fixed_split()
    say(f'Official split loaded in {time.perf_counter() - t:.2f} s: {len(ytr)} / {len(yval)}')
    xtr, ytr = select_stratified(xtr, ytr, args.train_size)
    xval, yval = select_stratified(xval, yval, args.val_size)
    say(f'Pilot train={len(ytr)} val={len(yval)}; X dtype={xtr.dtype}')
    hm = {w: load_writer(w) for w in WRITERS}
    com = BatchedCoM()
    t = time.perf_counter()
    centered_train = com.transform(xtr)
    centered_val = com.transform(xval)
    centered_hm = {w: (com.transform(hm[w][0]), hm[w][1]) for w in WRITERS}
    com_seconds = round(time.perf_counter() - t, 2)
    say('CoM preprocessed in', com_seconds, 'sec')
    if not np.isfinite(centered_train).all():
        raise RuntimeError('Nonfinite CoM transformed train')

    records = {}
    for label, train, val, hw in [
        ('baseline_subset', xtr, xval, {w: (hm[w][0], hm[w][1]) for w in WRITERS}),
        ('com_subset', centered_train, centered_val, centered_hm),
    ]:
        clf = pipeline(clone(svc), scaling)
        say(f'[{label}] fit START')
        t = time.perf_counter()
        clf.fit(train, ytr)
        fit_seconds = round(time.perf_counter() - t, 2)
        say(f'[{label}] fit DONE {fit_seconds}s')
        pv, predict_seconds = batched_predict(clf, val, label + ' official', args.batch_size)
        official = report(yval, pv)
        say(f'[{label}] official={official["accuracy"]:.6f}, F1={official["macro_f1"]:.6f} predict={predict_seconds}s')
        writers = []
        truths, preds = [], []
        for w in WRITERS:
            x, y = hw[w]
            p, pred_sec = batched_predict(clf, x, label + f' writer {w}', args.batch_size)
            m = report(y, p)
            writers.append({'writer': w, **m, 'predict_seconds': pred_sec})
            truths.append(y)
            preds.append(p)
            say(f'[{label}] writer {w}: {m["accuracy"]:.6f}')
        handmade = report(np.concatenate(truths), np.concatenate(preds))
        say(f'[{label}] Hand-made={handmade["accuracy"]:.6f} F1={handmade["macro_f1"]:.6f}')
        records[label] = {'fit_seconds': fit_seconds, 'official_validation': official,
                          'official_predict_seconds': predict_seconds,
                          'handmade_development': handmade, 'writers': writers}

    out = OUT_BASE / f'EXP_003_SVC_Normal_CoM_SAFE_PILOT_train{len(ytr)}'
    out.mkdir(parents=True, exist_ok=True)
    summary = {'experiment': 'EXP-003 CoM SAFE PILOT (subset, not comparable to EXP-001 full)',
               'train_size': len(ytr), 'validation_size': len(yval), 'com_preprocessing_seconds': com_seconds,
               'official_test_used': False, 'model_source': source, 'hyperparameters': svc.get_params(),
               'official_dataset_sha256': sha, 'results': records,
               'handmade_delta_pp': 100 * (records['com_subset']['handmade_development']['accuracy'] - records['baseline_subset']['handmade_development']['accuracy']),
               'official_delta_pp': 100 * (records['com_subset']['official_validation']['accuracy'] - records['baseline_subset']['official_validation']['accuracy'])}
    (out / 'metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    with (out / 'comparison.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['model', 'fit_seconds', 'official_accuracy', 'official_macro_f1', 'handmade_accuracy', 'handmade_macro_f1'])
        writer.writeheader()
        for name, r in records.items():
            writer.writerow({'model': name, 'fit_seconds': r['fit_seconds'],
                             'official_accuracy': r['official_validation']['accuracy'],
                             'official_macro_f1': r['official_validation']['macro_f1'],
                             'handmade_accuracy': r['handmade_development']['accuracy'],
                             'handmade_macro_f1': r['handmade_development']['macro_f1']})
    say('PILOT COMPARISON: CoM minus baseline official:', f'{summary["official_delta_pp"]:+.4f}pp',
        'Hand-made:', f'{summary["handmade_delta_pp"]:+.4f}pp')
    say('Saved:', out)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train-size', type=int, default=8000, choices=(2000, 4000, 8000, 16000, 24000))
    parser.add_argument('--val-size', type=int, default=3200, choices=(800, 1600, 3200, 6400))
    parser.add_argument('--batch-size', type=int, default=256, choices=(64, 128, 256, 512))
    run(parser.parse_args())


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")